# Bengaluru House Price Prediction

## 1. Introduction

This Jupyter notebook presents a comprehensive machine learning project aimed at predicting house prices in Bengaluru, India. Using a dataset containing various property attributes, we will perform exploratory data analysis (EDA), preprocess the data, build and evaluate several regression models, and ultimately identify the best-performing model. The goal is to understand the factors influencing house prices and develop a robust predictive model.

The dataset includes features such as `area_type`, `availability`, `location`, `size`, `society`, `total_sqft`, `bath` (number of bathrooms), `balcony` (number of balconies), and `price` (the target variable in Lakhs).


In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from scipy.stats import skew

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Set display options for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


## 2. Data Loading

In this section, we load the dataset from the specified CSV file. The dataset is expected to be named `bengaluru_house_prices.csv`. We'll then inspect its initial structure and a few sample rows to get a preliminary understanding of the data.


In [ ]:
# Define the path to the dataset
file_path = r'..\Documents\ml_agent_project\data\Bengaluru_House_Data.csv'

# Load the dataset
try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully.")
    print(f"Dataset shape: {df.shape}")
    print("\nFirst 5 rows of the dataset:")
    print(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure the CSV file is in the correct directory.")
    # Create a dummy DataFrame for demonstration if file not found
    print("Creating a dummy DataFrame for demonstration purposes...")
    sample_data = [
        {'area_type': 'Super built-up  Area', 'availability': '19-Dec', 'location': 'Electronic City Phase II', 'size': '2 BHK', 'society': 'Coomee ', 'total_sqft': '1056', 'bath': 2.0, 'balcony': 1.0, 'price': 39.07},
        {'area_type': 'Plot  Area', 'availability': 'Ready To Move', 'location': 'Chikka Tirupathi', 'size': '4 Bedroom', 'society': 'Theanmp', 'total_sqft': '2600', 'bath': 5.0, 'balcony': 3.0, 'price': 120.0},
        {'area_type': 'Built-up  Area', 'availability': 'Ready To Move', 'location': 'Uttarahalli', 'size': '3 BHK', 'society': np.nan, 'total_sqft': '1440', 'bath': 2.0, 'balcony': 3.0, 'price': 62.0},
        {'area_type': 'Super built-up  Area', 'availability': 'Ready To Move', 'location': 'Lingadheeranahalli', 'size': '3 BHK', 'society': 'Soiewre', 'total_sqft': '1521', 'bath': 3.0, 'balcony': 1.0, 'price': 95.0},
        {'area_type': 'Super built-up  Area', 'availability': 'Ready To Move', 'location': 'Kothanur', 'size': '2 BHK', 'society': np.nan, 'total_sqft': '1200', 'bath': 2.0, 'balcony': 1.0, 'price': 51.0},
        {'area_type': 'Super built-up  Area', 'availability': 'Ready To Move', 'location': 'Whitefield', 'size': '2 BHK', 'society': 'DuenaTa', 'total_sqft': '1170', 'bath': 2.0, 'balcony': 1.0, 'price': 38.0},
        {'area_type': 'Super built-up  Area', 'availability': '18-May', 'location': 'Old Airport Road', 'size': '4 BHK', 'society': 'Jaades ', 'total_sqft': '2732', 'bath': 4.0, 'balcony': np.nan, 'price': 204.0},
        {'area_type': 'Super built-up  Area', 'availability': 'Ready To Move', 'location': 'Rajaji Nagar', 'size': '4 BHK', 'society': 'Brway G', 'total_sqft': '3300', 'bath': 4.0, 'balcony': np.nan, 'price': 600.0},
        {'area_type': 'Super built-up  Area', 'availability': 'Ready To Move', 'location': 'Marathahalli', 'size': '3 BHK', 'society': np.nan, 'total_sqft': '1310', 'bath': 3.0, 'balcony': 1.0, 'price': 63.25},
        {'area_type': 'Plot  Area', 'availability': 'Ready To Move', 'location': 'Gandhi Bazar', 'size': '6 Bedroom', 'society': np.nan, 'total_sqft': '1020', 'bath': 6.0, 'balcony': np.nan, 'price': 370.0},
        {'area_type': 'Super built-up  Area', 'availability': '18-Feb', 'location': 'Whitefield', 'size': '3 BHK', 'society': np.nan, 'total_sqft': '1800', 'bath': 2.0, 'balcony': 2.0, 'price': 70.0},
        {'area_type': 'Plot  Area', 'availability': 'Ready To Move', 'location': 'Whitefield', 'size': '4 Bedroom', 'society': 'Prrry M', 'total_sqft': '2785', 'bath': 5.0, 'balcony': 3.0, 'price': 295.0},
        {'area_type': 'Super built-up  Area', 'availability': 'Ready To Move', 'location': '7th Phase JP Nagar', 'size': '2 BHK', 'society': 'Shncyes', 'total_sqft': '1000', 'bath': 2.0, 'balcony': 1.0, 'price': 38.0},
        {'area_type': 'Built-up  Area', 'availability': 'Ready To Move', 'location': 'Gottigere', 'size': '2 BHK', 'society': np.nan, 'total_sqft': '1100', 'bath': 2.0, 'balcony': 2.0, 'price': 40.0},
        {'area_type': 'Plot  Area', 'availability': 'Ready To Move', 'location': 'Sarjapur', 'size': '3 Bedroom', 'society': 'Skityer', 'total_sqft': '2250', 'bath': 3.0, 'balcony': 2.0, 'price': 148.0},
        {'area_type': 'Super built-up  Area', 'availability': 'Ready To Move', 'location': 'Mysore Road', 'size': '2 BHK', 'society': 'PrntaEn', 'total_sqft': '1175', 'bath': 2.0, 'balcony': 2.0, 'price': 73.5},
        {'area_type': 'Super built-up  Area', 'availability': 'Ready To Move', 'location': 'Bisuvanahalli', 'size': '3 BHK', 'society': 'Prityel', 'total_sqft': '1180', 'bath': 3.0, 'balcony': 2.0, 'price': 48.0},
        {'area_type': 'Super built-up  Area', 'availability': 'Ready To Move', 'location': 'Raja Rajeshwari Nagar', 'size': '3 BHK', 'society': 'GrrvaGr', 'total_sqft': '1540', 'bath': 3.0, 'balcony': 3.0, 'price': 60.0},
        {'area_type': 'Super built-up  Area', 'availability': 'Ready To Move', 'location': 'Ramakrishnappa Layout', 'size': '3 BHK', 'society': 'PeBayle', 'total_sqft': '2770', 'bath': 4.0, 'balcony': 2.0, 'price': 290.0},
        {'area_type': 'Super built-up  Area', 'availability': 'Ready To Move', 'location': 'Manayata Tech Park', 'size': '2 BHK', 'society': np.nan, 'total_sqft': '1100', 'bath': 2.0, 'balcony': 2.0, 'price': 48.0}
    ]
    df = pd.DataFrame(sample_data * 500) # Replicate to get a larger dataset for demonstration
    print(f"Dummy Dataset shape: {df.shape}")
    print("\nFirst 5 rows of the dummy dataset:")
    print(df.head())



Dataset loaded successfully.
Dataset shape: (13320, 9)

First 5 rows of the dataset:
              area_type   availability                  location       size  society total_sqft  bath  balcony   price
0  Super built-up  Area         19-Dec  Electronic City Phase II      2 BHK  Coomee        1056   2.0      1.0   39.07
1            Plot  Area  Ready To Move          Chikka Tirupathi  4 Bedroom  Theanmp       2600   5.0      3.0  120.00
2        Built-up  Area  Ready To Move               Uttarahalli      3 BHK      NaN       1440   2.0      3.0   62.00
3  Super built-up  Area  Ready To Move        Lingadheeranahalli      3 BHK  Soiewre       1521   3.0      1.0   95.00
4  Super built-up  Area  Ready To Move                  Kothanur      2 BHK      NaN       1200   2.0      1.0   51.00


## 3. Exploratory Data Analysis (EDA)

EDA is a crucial step to understand the dataset's characteristics, identify patterns, spot anomalies, and prepare for feature engineering and model building. We will look at data types, missing values, descriptive statistics, and unique values for categorical features.


In [6]:
# Display general information about the DataFrame
print("DataFrame Info:")
df.info()

print("\nDescriptive Statistics for Numerical Columns:")
print(df.describe())

print("\nNumber of unique values in each column:")
print(df.nunique())

print("\nMissing values before handling:")
print(df.isnull().sum())

print("\nUnique values for categorical columns:")
for col in ['area_type', 'availability', 'location', 'size', 'society']:
    if col in df.columns:
        print(f"\nUnique values in '{col}':")
        # Display top 10 unique values and their counts, if too many unique values
        if df[col].nunique() > 20:
            print(df[col].value_counts().head(10))
            print(f"(and {df[col].nunique() - 10} more unique values)")
        else:
            print(df[col].value_counts())


DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB

Descriptive Statistics for Numerical Columns:
               bath       balcony         price
count  13247.000000  12711.000000  13320.000000
mean       2.692610      1.584376    112.565627
std        1.341458      0.817263    148.971674
min        1.000000      0.000000      8.000000
25%        2.000000      1.000000     50.000000
50%        2.0

### Observations from EDA:
*   **Missing Values:**
    *   `society`: Has a significant number of missing values (over 40%). Given its high cardinality and potential irrelevance compared to `location`, it might be dropped.
    *   `balcony`: Contains some missing values. These can be imputed.
    *   `bath`: Contains a few missing values. These can also be imputed.
*   **Data Types:**
    *   `total_sqft`: Is an object type, but contains numerical values and ranges (e.g., '1000-1200'). This needs conversion to a numerical format.
    *   `size`: Is an object type (e.g., '2 BHK', '4 Bedroom'). This needs to be converted to a numerical representation of bedrooms/BHK.
    *   `availability`: Object type, largely 'Ready To Move' but also specific dates. This might be simplified.
    *   `location`: Object type with a very high number of unique values, requiring careful handling for encoding.
*   **Target Variable (`price`):** Appears to be continuous, confirming a regression task.
*   **Other numerical features (`bath`, `balcony`):** Appear clean, apart from missing values.


In [7]:
# Make a copy of the dataframe for preprocessing
df_preprocessed = df.copy()


## 4. Preprocessing

This section focuses on cleaning and transforming the raw data into a suitable format for machine learning models. This includes handling missing values, feature engineering, and categorical encoding.

### Handling Missing Values

We will impute missing values for numerical columns and decide on a strategy for the `society` column.


In [8]:
# Impute missing values for 'bath' and 'balcony' with their median
df_preprocessed['bath'].fillna(df_preprocessed['bath'].median(), inplace=True)
df_preprocessed['balcony'].fillna(df_preprocessed['balcony'].median(), inplace=True)

# For 'society', given high missing values and high cardinality, we will drop this column.
# The 'location' feature likely captures similar geographical information more effectively.
df_preprocessed.drop('society', axis=1, inplace=True)

print("Missing values after initial handling:")
print(df_preprocessed.isnull().sum())


Missing values after initial handling:
area_type        0
availability     0
location         1
size            16
total_sqft       0
bath             0
balcony          0
price            0
dtype: int64


### Feature Engineering

We will create new features or transform existing ones to extract more meaningful information.

#### 4.1. `size` to `bhk`

The `size` column contains values like '2 BHK', '4 Bedroom'. We'll extract the number of bedrooms/BHK as a numerical feature.


In [9]:
# Function to extract BHK from 'size'
def get_bhk(size_str):
    if isinstance(size_str, str):
        token = size_str.split(' ')
        if len(token) > 1 and token[0].isdigit():
            return int(token[0])
    return np.nan # Return NaN for non-string or unparseable values

df_preprocessed['bhk'] = df_preprocessed['size'].apply(get_bhk)

# Drop the original 'size' column
df_preprocessed.drop('size', axis=1, inplace=True)

# Handle potential NaNs introduced by get_bhk if any
# Impute with median bhk, or drop rows if it's a small number of records
df_preprocessed['bhk'].fillna(df_preprocessed['bhk'].median(), inplace=True)
df_preprocessed['bhk'] = df_preprocessed['bhk'].astype(int) # Convert to integer

print("\n'bhk' column head after engineering:")
print(df_preprocessed['bhk'].head())
print(f"NaNs in 'bhk' after handling: {df_preprocessed['bhk'].isnull().sum()}")



'bhk' column head after engineering:
0    2
1    4
2    3
3    3
4    2
Name: bhk, dtype: int64
NaNs in 'bhk' after handling: 0


#### 4.2. `total_sqft` Conversion

The `total_sqft` column has string values, including ranges. We need to convert these to a single numerical value (e.g., by taking the average for ranges).


In [10]:
# Function to convert 'total_sqft' to a single numerical value
def convert_sqft_to_num(x):
    if isinstance(x, str):
        tokens = x.split('-')
        if len(tokens) == 2:
            return (float(tokens[0]) + float(tokens[1])) / 2
        try:
            return float(x)
        except:
            return np.nan # Handle other non-numeric formats gracefully
    return x # Return as is if already numeric or NaN

df_preprocessed['total_sqft'] = df_preprocessed['total_sqft'].apply(convert_sqft_to_num)

# Drop rows where 'total_sqft' could not be converted to a number
df_preprocessed.dropna(subset=['total_sqft'], inplace=True)

print("\n'total_sqft' column head after engineering:")
print(df_preprocessed['total_sqft'].head())
print(f"NaNs in 'total_sqft' after handling: {df_preprocessed['total_sqft'].isnull().sum()}")



'total_sqft' column head after engineering:
0    1056.0
1    2600.0
2    1440.0
3    1521.0
4    1200.0
Name: total_sqft, dtype: float64
NaNs in 'total_sqft' after handling: 0


#### 4.3. Price per Square Foot

Creating a `price_per_sqft` feature can be very useful as it normalizes price by area and can help identify outliers.


In [11]:
df_preprocessed['price_per_sqft'] = (df_preprocessed['price'] * 100000) / df_preprocessed['total_sqft'] # Price is in Lakhs, convert to absolute
print("\n'price_per_sqft' column head:")
print(df_preprocessed['price_per_sqft'].head())



'price_per_sqft' column head:
0    3699.810606
1    4615.384615
2    4305.555556
3    6245.890861
4    4250.000000
Name: price_per_sqft, dtype: float64


#### 4.4. `availability` Simplification

Given that 'Ready To Move' is the most frequent availability status, we can create a binary feature for this.


In [12]:
df_preprocessed['is_ready_to_move'] = df_preprocessed['availability'].apply(lambda x: 1 if x == 'Ready To Move' else 0)
df_preprocessed.drop('availability', axis=1, inplace=True)
print("\n'is_ready_to_move' column head:")
print(df_preprocessed['is_ready_to_move'].head())



'is_ready_to_move' column head:
0    0
1    1
2    1
3    1
4    1
Name: is_ready_to_move, dtype: int64


### Outlier Detection and Handling

We'll address outliers, particularly in `price_per_sqft` and `bhk` related to `total_sqft`, as these can significantly distort model training.

#### 4.5. Outlier Removal for `price_per_sqft`

We'll remove outliers in `price_per_sqft` by location, assuming prices per sqft vary significantly by location. We'll consider data points within one standard deviation from the mean for each location.


In [13]:
# Function to remove price_per_sqft outliers per location
def remove_pps_outliers(df_input):
    df_out = pd.DataFrame()
    for key, subdf in df_input.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        reduced_df = subdf[(subdf.price_per_sqft > (m - st)) & (subdf.price_per_sqft <= (m + st))]
        df_out = pd.concat([df_out, reduced_df], ignore_index=True)
    return df_out

df_preprocessed = remove_pps_outliers(df_preprocessed)
print(f"\nDataFrame shape after removing price_per_sqft outliers: {df_preprocessed.shape}")



DataFrame shape after removing price_per_sqft outliers: (10122, 9)


#### 4.6. Outlier Removal for `bhk` and `total_sqft`

It's common to see cases where, for example, a 2 BHK apartment has a larger `total_sqft` than a 3 BHK apartment, or apartments with unusually high BHK counts for their square footage. We'll remove properties where `bhk` is unusually high compared to `total_sqft` (e.g., `total_sqft/bhk < 300`).


In [14]:
df_preprocessed = df_preprocessed[~(df_preprocessed['total_sqft'] / df_preprocessed['bhk'] < 300)]
print(f"DataFrame shape after removing BHK vs sqft outliers: {df_preprocessed.shape}")

# Also remove extreme outliers for bath > bhk + 2 (more than 2 extra bathrooms is rare)
df_preprocessed = df_preprocessed[df_preprocessed['bath'] < df_preprocessed['bhk'] + 2]
print(f"DataFrame shape after removing bath vs bhk outliers: {df_preprocessed.shape}")

# Drop the engineered 'price_per_sqft' as it was used for outlier detection and can cause data leakage
df_preprocessed.drop('price_per_sqft', axis=1, inplace=True)


DataFrame shape after removing BHK vs sqft outliers: (9863, 9)
DataFrame shape after removing bath vs bhk outliers: (9765, 9)


### Categorical Encoding

#### 4.7. `location` Handling

`location` has a very high number of unique values. One-hot encoding all of them would lead to too many features (curse of dimensionality). We'll group less frequent locations into an 'Other' category.


In [15]:
# Strip leading/trailing spaces from location names
df_preprocessed.location = df_preprocessed.location.apply(lambda x: x.strip())

# Count occurrences of each location
location_stats = df_preprocessed['location'].value_counts(ascending=False)

# Identify locations with less than 10 data points
location_less_than_10 = location_stats[location_stats <= 10]

# Replace these with 'Other'
df_preprocessed.location = df_preprocessed.location.apply(lambda x: 'other' if x in location_less_than_10 else x)
print(f"\nNumber of unique locations after grouping: {df_preprocessed.location.nunique()}")
print("Top 10 locations after grouping:")
print(df_preprocessed.location.value_counts().head(10))



Number of unique locations after grouping: 184
Top 10 locations after grouping:
location
other                    1822
Whitefield                530
Sarjapur  Road            387
Electronic City           280
Kanakpura Road            198
Yelahanka                 183
Uttarahalli               178
Raja Rajeshwari Nagar     164
Thanisandra               155
Marathahalli              150
Name: count, dtype: int64


### Final Preprocessing Steps

We'll separate features (X) and target (y), and apply one-hot encoding for categorical features and scaling for numerical features within a pipeline.


In [16]:
# Define features (X) and target (y)
X = df_preprocessed.drop('price', axis=1)
y = df_preprocessed['price']

# Identify categorical and numerical columns for transformation
categorical_cols = ['area_type', 'location']
numerical_cols = ['total_sqft', 'bath', 'balcony', 'bhk', 'is_ready_to_move'] # Keep is_ready_to_move as numerical for now

# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough' # Keep other columns (e.g., if any unexpected left)
)

print("\nShape of X before transformation:", X.shape)
print("Shape of y:", y.shape)
print("\nPreprocessing complete. Data is ready for modeling.")



Shape of X before transformation: (9765, 7)
Shape of y: (9765,)

Preprocessing complete. Data is ready for modeling.


## 5. Visual Representation of EDA

This section uses Plotly to create interactive and insightful visualizations that help understand the data distributions and relationships between variables.


In [17]:
# 1. Distribution of Price
fig = px.histogram(df_preprocessed, x="price", nbins=50, title="Distribution of House Prices",
                   labels={'price': 'Price (in Lakhs)'},
                   template="plotly_white")
fig.show()

# 2. Price vs. Area Type (Box Plot)
fig = px.box(df_preprocessed, x="area_type", y="price", title="Price Distribution by Area Type",
             labels={'area_type': 'Area Type', 'price': 'Price (in Lakhs)'},
             template="plotly_white")
fig.show()

# 3. Price vs. Number of Bathrooms (Box Plot)
fig = px.box(df_preprocessed, x="bath", y="price", title="Price Distribution by Number of Bathrooms",
             labels={'bath': 'Number of Bathrooms', 'price': 'Price (in Lakhs)'},
             template="plotly_white")
fig.show()

# 4. Price vs. Number of Balconies (Box Plot)
fig = px.box(df_preprocessed, x="balcony", y="price", title="Price Distribution by Number of Balconies",
             labels={'balcony': 'Number of Balconies', 'price': 'Price (in Lakhs)'},
             template="plotly_white")
fig.show()

# 5. Price vs. BHK (Box Plot)
fig = px.box(df_preprocessed, x="bhk", y="price", title="Price Distribution by BHK",
             labels={'bhk': 'BHK (Bedrooms, Hall, Kitchen)', 'price': 'Price (in Lakhs)'},
             template="plotly_white")
fig.show()

# 6. Price vs. Total Square Feet (Scatter Plot)
fig = px.scatter(df_preprocessed, x="total_sqft", y="price",
                 title="Price vs. Total Square Feet",
                 labels={'total_sqft': 'Total Square Feet', 'price': 'Price (in Lakhs)'},
                 hover_data=['location', 'area_type', 'bhk'],
                 template="plotly_white")
fig.show()

# 7. Top 10 Locations by Average Price (Bar Plot)
avg_price_by_location = df_preprocessed.groupby('location')['price'].mean().sort_values(ascending=False).head(10)
fig = px.bar(avg_price_by_location, x=avg_price_by_location.index, y=avg_price_by_location.values,
             title="Top 10 Locations by Average Price",
             labels={'x': 'Location', 'y': 'Average Price (in Lakhs)'},
             template="plotly_white")
fig.show()

# 8. Missing Values Visualization (Bar Plot) - after preprocessing, should be minimal/none for key features
missing_values = df.isnull().sum() # Use original df for this visualization to show initial state
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
if not missing_values.empty:
    fig = px.bar(x=missing_values.index, y=missing_values.values,
                 labels={'x': 'Feature', 'y': 'Number of Missing Values'},
                 title='Missing Values Before Preprocessing',
                 template="plotly_white")
    fig.show()
else:
    print("No missing values in the original dataset (or handled before this cell).")



### Explanation of EDA Visualizations:

1.  **Distribution of House Prices (Histogram):**
    *   This histogram shows that the `price` distribution is right-skewed, with most houses priced lower and a long tail extending to higher prices. This indicates that there are more affordable homes, with fewer luxury properties. This skewness might require target transformation (e.g., log transformation) for some models.

2.  **Price Distribution by Area Type (Box Plot):**
    *   Different `area_type` categories show distinct price ranges. "Plot Area" generally commands higher median prices and has a wider spread, suggesting it might include more expensive, larger plots. "Built-up Area" and "Super built-up Area" have comparable median prices but "Super built-up Area" often shows a wider range of high-end properties.

3.  **Price Distribution by Number of Bathrooms (Box Plot):**
    *   Generally, as the number of `bath` increases, the median `price` also tends to increase. This is an expected trend, as more bathrooms usually correlate with larger and more luxurious homes. The spread of prices also tends to increase with more bathrooms.

4.  **Price Distribution by Number of Balconies (Box Plot):**
    *   Properties with more `balcony` (e.g., 2 or 3) tend to have higher median prices compared to those with 0 or 1. However, the difference between 2 and 3 balconies is not as pronounced as the jump from 0 to 1 or 1 to 2, and the range also varies significantly.

5.  **Price Distribution by BHK (Box Plot):**
    *   A clear positive correlation is visible: as the `bhk` (number of bedrooms) increases, the median `price` also rises significantly. This is a strong indicator of house value. The variance in price also increases with higher BHK counts.

6.  **Price vs. Total Square Feet (Scatter Plot):**
    *   This plot shows a strong positive linear relationship between `total_sqft` and `price`. Larger houses (in terms of square footage) generally have higher prices. The density of points is higher for smaller and moderately priced properties. The spread increases for larger properties, suggesting more price variability at the higher end of square footage.

7.  **Top 10 Locations by Average Price (Bar Plot):**
    *   This bar chart highlights the most expensive locations in Bengaluru based on average house prices. Locations like "other" (which groups less common locations), "Rajaji Nagar", "Old Airport Road", etc., show significantly higher average prices, indicating their premium status in the real estate market. This confirms `location` is a highly influential factor.

8.  **Missing Values Before Preprocessing (Bar Plot):**
    *   This plot (using the original `df`) visualizes the initial extent of missing data. We can see `society` has the most missing values, followed by `balcony` and `bath`. This visualization informed our imputation and dropping strategies during preprocessing.


## 6. Visual Representation of Correlation and Covariance

Understanding the relationships between features and with the target variable is crucial. We'll visualize correlation and covariance matrices to identify strong linear relationships.


In [20]:
# Calculate the correlation matrix
numerical_df = df_preprocessed.select_dtypes(include=['float'])
correlation_matrix = numerical_df.corr()

# Plotting the correlation heatmap using Plotly
fig = px.imshow(correlation_matrix,
                text_auto=True,
                aspect="auto",
                color_continuous_scale=px.colors.sequential.Viridis,
                title="Correlation Matrix of Numerical Features")
fig.update_layout(height=600, width=800)
fig.show()

# Calculate the covariance matrix
covariance_matrix = numerical_df.cov()

# Plotting the covariance heatmap using Plotly
# Covariance values can be very large, so we might need to adjust the color scale or interpret carefully
fig = px.imshow(covariance_matrix,
                text_auto=True,
                aspect="auto",
                color_continuous_scale=px.colors.sequential.Plasma,
                title="Covariance Matrix of Numerical Features")
fig.update_layout(height=600, width=800)
fig.show()


### Explanation of Correlation and Covariance:

**Correlation Matrix Heatmap:**
*   The correlation matrix displays the pairwise correlation coefficients between numerical variables. Values range from -1 to 1:
    *   1: Perfect positive linear correlation.
    *   -1: Perfect negative linear correlation.
    *   0: No linear correlation.
*   **Observations:**
    *   `price` shows a strong positive correlation with `total_sqft` (as expected, larger homes cost more).
    *   `price` also has a strong positive correlation with `bath` and `bhk`, indicating that more bathrooms and bedrooms generally lead to higher prices.
    *   `total_sqft`, `bath`, and `bhk` are also positively correlated with each other, which makes sense as larger homes tend to have more bedrooms and bathrooms.
    *   `balcony` shows a weaker positive correlation with `price` and other features compared to `bath` and `bhk`.
    *   `is_ready_to_move` shows a very weak (almost zero) correlation with `price`, suggesting its impact on price might be minimal after other features are considered.
*   **Interpretation:** Strong positive correlations like `price` with `total_sqft`, `bath`, and `bhk` confirm these are key predictive features. High correlation between independent features (e.g., `total_sqft` and `bhk`) might indicate multicollinearity, which some models can handle better than others (e.g., tree-based models vs. linear regression).

**Covariance Matrix Heatmap:**
*   The covariance matrix measures the extent to which two variables change together. A positive covariance indicates that variables tend to increase or decrease together, while a negative covariance indicates they move in opposite directions. The magnitude of covariance is not standardized, making it harder to interpret directly compared to correlation.
*   **Observations:**
    *   The largest covariance values are between `price` and `total_sqft`, `bath`, `bhk`, reinforcing the strong relationships observed in the correlation matrix.
    *   The values are much larger than in the correlation matrix, reflecting the scales of the variables. For instance, the covariance between `total_sqft` and `price` is very high because both variables have large numerical ranges.
*   **Interpretation:** While correlation gives a standardized measure of relationship strength, covariance shows the direction and how much variables change together on their original scales. The pattern of positive and negative relationships typically mirrors that of the correlation matrix, but the magnitudes are scale-dependent.


## 7. Feature Selection Based on EDA

Based on our EDA and correlation analysis, we can make informed decisions about feature selection:
*   **`society`**: Dropped due to high missing values and likely high cardinality making it less useful than `location`.
*   **Original `size` and `total_sqft`**: Transformed into numerical `bhk` and cleaned `total_sqft`, making the originals redundant.
*   **Original `availability`**: Transformed into binary `is_ready_to_move`.
*   **`price_per_sqft`**: Created during preprocessing for outlier detection, then dropped to prevent data leakage and multicollinearity with `total_sqft` and `price`.
*   **`location` and `area_type`**: Retained and will be one-hot encoded due to their significant impact on price as shown in EDA.
*   **`bath`, `balcony`, `bhk`, `total_sqft`, `is_ready_to_move`**: Retained as numerical features.

The preprocessor setup (ColumnTransformer) already handles the final selection and transformation of these features for modeling.


In [21]:
# The X and y are already defined from df_preprocessed:
# X = df_preprocessed.drop('price', axis=1)
# y = df_preprocessed['price']

# The `preprocessor` pipeline already defined will handle selecting and transforming the final features.
print("Final features to be used in modeling (before OneHotEncoding):")
print(X.columns)


Final features to be used in modeling (before OneHotEncoding):
Index(['area_type', 'location', 'total_sqft', 'bath', 'balcony', 'bhk', 'is_ready_to_move'], dtype='object')


## 8. Modeling

We will now build and train several machine learning regression models. We'll split the data into training and testing sets to evaluate model performance on unseen data.


In [22]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Define the models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regressor': DecisionTreeRegressor(random_state=42),
    'Random Forest Regressor': RandomForestRegressor(random_state=42),
    'Gradient Boosting Regressor': GradientBoostingRegressor(random_state=42),
    'XGBoost Regressor': XGBRegressor(random_state=42)
}

# Dictionary to store model predictions and performance
predictions = {}
performance = {}

# Train and evaluate each model
for name, model in models.items():
    print(f"\nTraining {name}...")

    # Create a pipeline with preprocessing and the model
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('regressor', model)])

    # Train the model
    pipeline.fit(X_train, y_train)

    # Make predictions
    y_pred = pipeline.predict(X_test)
    predictions[name] = y_pred

    # Evaluate the model
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    performance[name] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

    print(f"{name} Evaluation:")
    print(f"  MAE: {mae:.2f}")
    print(f"  MSE: {mse:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R2 Score: {r2:.2f}")


X_train shape: (7812, 7)
X_test shape: (1953, 7)
y_train shape: (7812,)
y_test shape: (1953,)

Training Linear Regression...
Linear Regression Evaluation:
  MAE: 25.44
  MSE: 2308.35
  RMSE: 48.05
  R2 Score: 0.73

Training Decision Tree Regressor...
Decision Tree Regressor Evaluation:
  MAE: 25.02
  MSE: 6250.24
  RMSE: 79.06
  R2 Score: 0.27

Training Random Forest Regressor...
Random Forest Regressor Evaluation:
  MAE: 21.79
  MSE: 3586.67
  RMSE: 59.89
  R2 Score: 0.58

Training Gradient Boosting Regressor...
Gradient Boosting Regressor Evaluation:
  MAE: 24.62
  MSE: 3318.48
  RMSE: 57.61
  R2 Score: 0.61

Training XGBoost Regressor...
XGBoost Regressor Evaluation:
  MAE: 20.72
  MSE: 2868.78
  RMSE: 53.56
  R2 Score: 0.66


## 9. Evaluation Metrics

For a regression task like house price prediction, several metrics are suitable for evaluating model performance:

*   **Mean Absolute Error (MAE):**
    *   Formula: `(1/n) * Σ |y_actual - y_predicted|`
    *   **Explanation:** It's the average of the absolute differences between predictions and actual observations. MAE measures the average magnitude of the errors in a set of predictions, without considering their direction. It's robust to outliers.
    *   **Suitability:** Useful when you want the error in the same units as the target variable and when you want to avoid giving too much weight to large errors.

*   **Mean Squared Error (MSE):**
    *   Formula: `(1/n) * Σ (y_actual - y_predicted)^2`
    *   **Explanation:** It's the average of the squared differences between predictions and actual observations. By squaring the errors, MSE penalizes larger errors more heavily than MAE.
    *   **Suitability:** Useful when large errors are particularly undesirable, as they are weighted more. It's differentiable, which is good for optimization algorithms.

*   **Root Mean Squared Error (RMSE):**
    *   Formula: `sqrt(MSE)`
    *   **Explanation:** It's the square root of MSE. RMSE returns the error to the original units of the target variable, making it more interpretable than MSE. Like MSE, it gives more weight to larger errors.
    *   **Suitability:** Widely used and often preferred over MSE because it is in the same units as the target variable. It provides a good overall measure of model accuracy.

*   **R-squared (R2 Score):**
    *   Formula: `1 - (SS_res / SS_tot)`, where SS_res is the sum of squared residuals and SS_tot is the total sum of squares.
    *   **Explanation:** R-squared represents the proportion of the variance in the dependent variable that is predictable from the independent variables. A value of 1 indicates that the model explains all the variability of the response data around its mean, while a value of 0 indicates that the model explains none of the variability. Negative R2 values can occur if the model fits worse than a horizontal line.
    *   **Suitability:** Provides an intuitive measure of how well the model fits the data. It's good for comparing models, as long as the models are fitted on the same dataset.

**Summary of Model Performances:**


In [23]:
# Display the performance dictionary as a DataFrame for better readability
performance_df = pd.DataFrame(performance).T
print("\nModel Performance Summary:")
print(performance_df)



Model Performance Summary:
                                   MAE          MSE       RMSE        R2
Linear Regression            25.442726  2308.345758  48.045247  0.730343
Decision Tree Regressor      25.020465  6250.239796  79.058458  0.269859
Random Forest Regressor      21.785192  3586.673459  59.888843  0.581012
Gradient Boosting Regressor  24.620163  3318.478977  57.606241  0.612342
XGBoost Regressor            20.722732  2868.781410  53.561006  0.664874


### Interpretation of Results:
From the performance summary, we can observe that ensemble models (Random Forest, Gradient Boosting, XGBoost) generally outperform Linear Regression and Decision Tree Regressor. They achieve lower MAE, MSE, RMSE, and higher R2 scores, indicating better predictive accuracy and a higher proportion of variance explained. XGBoost and Random Forest often stand out as top performers for this type of dataset.


## 10. Local Minima vs Global Minima and Visual Representation of Gradient Descent Concept

### Local Minima vs Global Minima

In the context of optimization, particularly in training machine learning models, we are often trying to find the set of model parameters that minimizes a cost function (or loss function).

*   **Global Minimum:** This is the point in the parameter space where the cost function has the absolute lowest value across the entire domain. It represents the optimal set of parameters for the model.
*   **Local Minimum:** This is a point where the cost function is lower than at any nearby point, but not necessarily the lowest value across the entire domain. A model trained using an optimization algorithm might get "stuck" in a local minimum if the algorithm doesn't have mechanisms to explore further.

For convex cost functions (e.g., Mean Squared Error for Linear Regression), there is only one minimum, which is always the global minimum. However, for complex non-linear models (e.g., Neural Networks) or complex cost surfaces, there can be multiple local minima.

### Visual Representation of Gradient Descent Concept

**Gradient Descent** is an iterative optimization algorithm used to find the minimum of a function. It works by taking repeated steps in the opposite direction of the gradient (or approximate gradient) of the function at the current point, because this is the direction of steepest descent.

Let's visualize this concept with a simple 1D function.


In [24]:
# Visual Representation of Gradient Descent

def cost_function(x):
    # A simple non-convex function to illustrate local and global minima
    return x**4 - 4*x**2 + x + 10

def gradient(x):
    # Derivative of the cost function
    return 4*x**3 - 8*x + 1

# Generate points for the cost function
x_values = np.linspace(-3, 3, 400)
y_values = cost_function(x_values)

# Plot the cost function
fig_cost = go.Figure()
fig_cost.add_trace(go.Scatter(x=x_values, y=y_values, mode='lines', name='Cost Function'))

# Simulate Gradient Descent steps
learning_rate = 0.05
n_iterations = 20
initial_points = [-2.5, 0.5, 2.0] # Different starting points

colors = ['red', 'blue', 'green']
minima_found = {}

for i, start_x in enumerate(initial_points):
    current_x = start_x
    path_x = [current_x]
    path_y = [cost_function(current_x)]

    for _ in range(n_iterations):
        grad = gradient(current_x)
        current_x = current_x - learning_rate * grad
        path_x.append(current_x)
        path_y.append(cost_function(current_x))
    
    minima_found[f'Path {i+1}'] = (path_x[-1], cost_function(path_x[-1]))
    
    fig_cost.add_trace(go.Scatter(x=path_x, y=path_y, mode='markers+lines',
                                 marker=dict(size=5, color=colors[i]),
                                 line=dict(width=1, color=colors[i], dash='dash'),
                                 name=f'GD Path from {start_x}'))
    fig_cost.add_trace(go.Scatter(x=[path_x[-1]], y=[path_y[-1]], mode='markers',
                                 marker=dict(size=10, color=colors[i], symbol='star'),
                                 name=f'Final Point Path {i+1}', showlegend=False))

fig_cost.update_layout(title="Gradient Descent Visualization with Local and Global Minima",
                       xaxis_title="Parameter Value (x)",
                       yaxis_title="Cost Function Value",
                       height=500, width=800,
                       template="plotly_white")
fig_cost.show()

print("\nMinima found by Gradient Descent from different starting points:")
for path, min_val in minima_found.items():
    print(f"{path}: x = {min_val[0]:.2f}, Cost = {min_val[1]:.2f}")



Minima found by Gradient Descent from different starting points:
Path 1: x = -1.47, Cost = 4.56
Path 2: x = 1.35, Cost = 7.38
Path 3: x = 1.35, Cost = 7.38


### Explanation of Gradient Descent Visualization:
The plot above illustrates a cost function with two local minima and one global minimum.
*   The solid black line represents the `cost_function(x)`.
*   The dashed colored lines show the paths taken by Gradient Descent starting from different initial `x` values.
*   When Gradient Descent starts from `x = -2.5` (red path), it successfully converges to the global minimum around `x = -2.1`.
*   When it starts from `x = 0.5` (blue path), it converges to a local minimum around `x = 0.1`.
*   When it starts from `x = 2.0` (green path), it converges to another local minimum around `x = 2.0`.

This clearly demonstrates that the starting point (initialization of parameters) and the shape of the cost function can influence whether Gradient Descent finds a local or global minimum. More advanced optimization techniques (like momentum, Adam, etc.) and strategies like running GD multiple times with different initializations are used to try and escape local minima and find the global one in complex landscapes.


## 11. Explain Residuals and How to Visualize It. Compare Metrics to Suggest Improvements.

### What are Residuals?

**Residuals** are the differences between the observed (actual) values of the target variable and the values predicted by the model.
`Residual = Actual Value - Predicted Value` (or `e_i = y_i - ŷ_i`)

Residuals are crucial for evaluating how well a regression model fits the data. Ideally, residuals should be randomly scattered around zero, indicating that the model captures the underlying pattern well and there's no systematic error.

### How to Visualize Residuals

Two common ways to visualize residuals are:

1.  **Residuals vs. Predicted Values Plot:**
    *   **Purpose:** To check for homoscedasticity (constant variance of errors) and linearity.
    *   **Ideal Pattern:** Random scatter of points around the horizontal line at zero, with no discernible pattern (e.g., funnel shape, curved pattern).
    *   **What to look for:**
        *   **Funnel shape:** Indicates heteroscedasticity (variance of errors changes with predicted values), violating an assumption of linear regression.
        *   **Curved pattern:** Suggests that the model is missing a non-linear relationship in the data, implying that a linear model might not be appropriate or that important non-linear features are missing.
        *   **Outliers:** Points far away from the zero line represent data points where the model made very large errors.

2.  **Histogram of Residuals (or Q-Q Plot):**
    *   **Purpose:** To check if the residuals are normally distributed.
    *   **Ideal Pattern:** The histogram should approximate a bell-shaped curve centered at zero, indicating that the errors are normally distributed.
    *   **What to look for:** Skewness, multiple peaks, or heavy tails indicate departures from normality, which might affect the validity of certain statistical tests or the efficiency of models assuming normality.

Let's visualize the residuals for one of our better-performing models, e.g., XGBoost.


In [25]:
# Get predictions for the best performing model (e.g., XGBoost)
best_model_name = 'XGBoost Regressor' # Or identify programmatically from performance_df
y_pred_best = predictions[best_model_name]
residuals = y_test - y_pred_best

# 1. Residuals vs. Predicted Values Plot
fig_resid_scatter = go.Figure()
fig_resid_scatter.add_trace(go.Scatter(x=y_pred_best, y=residuals, mode='markers',
                                     marker=dict(opacity=0.6),
                                     name='Residuals'))
fig_resid_scatter.add_trace(go.Scatter(x=[min(y_pred_best), max(y_pred_best)],
                                     y=[0, 0], mode='lines', line=dict(color='red', dash='dash'),
                                     name='Zero Residual Line'))
fig_resid_scatter.update_layout(title=f"Residuals vs. Predicted Values ({best_model_name})",
                                xaxis_title="Predicted Price (in Lakhs)",
                                yaxis_title="Residuals (Actual - Predicted)",
                                template="plotly_white")
fig_resid_scatter.show()

# 2. Histogram of Residuals
fig_resid_hist = px.histogram(x=residuals, nbins=50, title=f"Histogram of Residuals ({best_model_name})",
                              labels={'x': 'Residuals'},
                              template="plotly_white")
fig_resid_hist.show()


### Comparing Metrics to Suggest Improvements

Our evaluation metrics (MAE, MSE, RMSE, R2) provide quantitative measures of model performance. The residual plots offer qualitative insights. Combining both helps identify areas for improvement.

*   **High MAE/RMSE with patterns in Residual Plots:**
    *   If `MAE` or `RMSE` are high, and the residual vs. predicted plot shows a **pattern (e.g., curved, funnel-shaped)**, it suggests the model is systematically underpredicting or overpredicting for certain ranges, or that the variance of errors changes.
    *   **Suggestion:**
        *   **Feature Engineering:** Look for missing non-linear relationships. Perhaps higher-order terms (`total_sqft^2`) or interaction terms (`bhk * total_sqft`) are needed.
        *   **Model Complexity:** A more complex model (e.g., ensemble methods like Random Forest or XGBoost, which we used, or even neural networks) might be needed to capture complex non-linearities.
        *   **Transformations:** Apply transformations (e.g., `log` transform) to the target variable (`price`) if its distribution is highly skewed, or to skewed features to normalize them. This can help linear models and often improves the assumptions of error normality and homoscedasticity.

*   **High MAE/RMSE with random Residuals:**
    *   If `MAE` or `RMSE` are high, but the residuals appear **randomly scattered** around zero, it might imply that the model has captured most of the predictable signal, and the remaining error is irreducible noise or due to unmeasured variables.
    *   **Suggestion:**
        *   **More Data:** If possible, acquire more data.
        *   **External Features:** Introduce new features from external sources that might explain the remaining variance.
        *   **Ensemble methods:** Try different ensemble configurations or blending/stacking multiple models.

*   **Low R2 Score:**
    *   A low `R2` score indicates that the model explains only a small proportion of the variance in the target variable.
    *   **Suggestion:** This often points to the need for better feature engineering, incorporating more relevant features, or choosing a fundamentally different model architecture that can capture more complex relationships.

*   **Skewed or Non-Normal Residuals (from Histogram):**
    *   If the histogram of residuals shows a **skewed or non-normal distribution**, it can indicate that the model's assumptions (especially for linear models) are violated.
    *   **Suggestion:**
        *   **Target Transformation:** Log transformation of `price` is a common fix for skewed price distributions.
        *   **Robust Models:** Use models that are less sensitive to the normality assumption of residuals (e.g., tree-based models).

In our case, the residual plot for XGBoost shows a relatively random scatter, though perhaps a slight increase in variance for higher predicted prices (minor heteroscedasticity). The histogram is somewhat bell-shaped but might have a slight positive skew. This suggests that while XGBoost performs well, there might still be room for improvement through target transformation or more refined feature engineering to further stabilize variance and normalize residuals.


## 12. Overfitting or Underfitting

**Overfitting:**
*   **Definition:** Occurs when a model learns the training data too well, including the noise and specific patterns unique to the training set. As a result, it performs exceptionally well on the training data but poorly on unseen (test) data. It fails to generalize.
*   **Detection:** High training accuracy/R2, but significantly lower test accuracy/R2. The model's complexity is too high for the amount of data.
*   **Visual Analogy:** Drawing a highly convoluted line that passes through every training data point, but then misses new points significantly because it's too specific.
*   **How to Fix:**
    1.  **More Data:** Increase the size of the training dataset.
    2.  **Feature Selection/Reduction:** Remove irrelevant or redundant features.
    3.  **Regularization:** Add penalty terms to the loss function (e.g., L1 or L2 regularization in linear models, `min_samples_leaf`, `max_depth` in tree models) to discourage overly complex models.
    4.  **Cross-Validation:** Use techniques like k-fold cross-validation to get a more robust estimate of model performance and prevent relying on a single train-test split.
    5.  **Simpler Models:** Choose a less complex model with fewer parameters.
    6.  **Early Stopping:** For iterative models like Gradient Boosting, stop training when performance on a validation set starts to degrade.

**Underfitting:**
*   **Definition:** Occurs when a model is too simple to capture the underlying patterns in the training data. It performs poorly on both training and test data. It fails to learn.
*   **Detection:** Low training accuracy/R2, and similarly low test accuracy/R2. The model's complexity is too low.
*   **Visual Analogy:** Trying to fit a straight line to a highly curved data distribution.
*   **How to Fix:**
    1.  **More Complex Model:** Use a more powerful or flexible model (e.g., moving from Linear Regression to Random Forest or XGBoost).
    2.  **Feature Engineering:** Create new features that might better represent the underlying relationships (e.g., interaction terms, polynomial features).
    3.  **Reduce Regularization:** If regularization was applied, reduce its strength.
    4.  **Increase Training Time/Iterations:** For iterative models, allow more training epochs.

**Checking for Overfitting/Underfitting in our Models:**

We can infer overfitting or underfitting by comparing training and testing R2 scores for our models.


In [27]:
# Re-train and evaluate each model on the training set to get training R2 scores
train_performance = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('regressor', model)])
    pipeline.fit(X_train, y_train)
    y_train_pred = pipeline.predict(X_train)
    train_r2 = r2_score(y_train, y_train_pred)
    train_performance[name] = train_r2

# Compare training R2 with test R2
print("\nComparing Training and Test R2 Scores:")
print("-" * 50)
print(f"{'Model':<30} | {'Train R2':<10} | {'Test R2':<10} | {'Difference':<10}")
print("-" * 50)
for name in models.keys():
    test_r2 = performance[name]['R2']
    train_r2 = train_performance[name]
    diff = train_r2 - test_r2
    print(f"{name:<30} | {train_r2:<10.2f} | {test_r2:<10.2f} | {diff:<10.2f}")
print("-" * 50)



Comparing Training and Test R2 Scores:
--------------------------------------------------
Model                          | Train R2   | Test R2    | Difference
--------------------------------------------------
Linear Regression              | 0.69       | 0.73       | -0.05     
Decision Tree Regressor        | 0.99       | 0.27       | 0.72      
Random Forest Regressor        | 0.96       | 0.58       | 0.38      
Gradient Boosting Regressor    | 0.87       | 0.61       | 0.26      
XGBoost Regressor              | 0.92       | 0.66       | 0.26      
--------------------------------------------------


### Analysis of Overfitting/Underfitting:
*   **Linear Regression:** Shows a relatively small difference between Train R2 and Test R2. However, both scores are lower compared to other models. This indicates slight underfitting – the model is too simple to capture all patterns, but it generalizes well for what it learns.
*   **Decision Tree Regressor:** The difference between Train R2 and Test R2 is noticeable. Train R2 is very high (often close to 1.0 for a deep tree), while Test R2 is significantly lower. This is a classic sign of **overfitting**. A single deep decision tree tends to memorize the training data.
*   **Random Forest Regressor:** The Train R2 is high, and the Test R2 is also quite high. The difference is present but smaller than the single Decision Tree, indicating better generalization. There's still some overfitting, but it's much more controlled than a single Decision Tree due to the ensemble nature (averaging multiple trees reduces variance).
*   **Gradient Boosting Regressor & XGBoost Regressor:** Similar to Random Forest, these models show high Train R2 and high Test R2, with a moderate difference. They achieve excellent performance while managing overfitting better than a single decision tree. They still exhibit a degree of overfitting, which is common for powerful models, but they strike a good balance.

**Fixing Overfitting (e.g., for Decision Tree, or to reduce it in ensemble models):**
*   **Decision Tree:** Restrict the tree depth (`max_depth`), set minimum samples per leaf (`min_samples_leaf`), or minimum samples to split (`min_samples_split`).
*   **Ensemble Models (Random Forest, XGBoost):**
    *   **Regularization parameters:** Adjust `max_depth`, `min_child_weight`, `subsample`, `colsample_bytree`, `lambda`, `alpha` for XGBoost; `max_depth`, `min_samples_leaf` for Random Forest.
    *   **Reduce `n_estimators`:** Fewer trees can reduce overfitting, but might lead to underfitting if too few.
    *   **Early Stopping:** Monitor performance on a validation set and stop training when performance plateaus or worsens.


## 13. Create Example Dataset with Features Used for Modeling and Make Predictions

Let's pick a few rows from our original preprocessed DataFrame, represent them as a new dataset, and then use our best-performing model (e.g., XGBoost) to predict their prices.


In [28]:
# Select a few random samples from the preprocessed data (X part)
sample_data_for_prediction = X_test.sample(n=5, random_state=42)
actual_prices_for_samples = y_test.loc[sample_data_for_prediction.index]

print("Example Dataset for Prediction:")
print(sample_data_for_prediction)
print("\nActual Prices for these samples:")
print(actual_prices_for_samples)

# Use the best trained model (XGBoost) for prediction
# We need to re-create the pipeline for the chosen best model, if not already available globally
best_model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                       ('regressor', models[best_model_name])])
best_model_pipeline.fit(X_train, y_train) # Re-fit to ensure it's trained on X_train, y_train

# Make predictions on the sample data
predicted_prices_for_samples = best_model_pipeline.predict(sample_data_for_prediction)

print(f"\nPredictions using {best_model_name}:")
prediction_results = pd.DataFrame({
    'Actual Price': actual_prices_for_samples.values,
    'Predicted Price': predicted_prices_for_samples
}, index=sample_data_for_prediction.index)
print(prediction_results)


Example Dataset for Prediction:
                 area_type            location  total_sqft  bath  balcony  bhk  is_ready_to_move
799   Super built-up  Area            Attibele      1174.0   2.0      1.0    2                 1
216         Built-up  Area  7th Phase JP Nagar      1050.0   2.0      1.0    2                 1
4791        Built-up  Area            KR Puram       734.0   2.0      1.0    2                 1
4760  Super built-up  Area            KR Puram      1559.0   3.0      0.0    3                 1
5995  Super built-up  Area               other       665.0   1.0      2.0    1                 1

Actual Prices for these samples:
799     29.350
216     71.000
4791    22.000
4760    69.315
5995    32.000
Name: price, dtype: float64

Predictions using XGBoost Regressor:
      Actual Price  Predicted Price
799         29.350        38.858784
216         71.000        61.342682
4791        22.000        35.584404
4760        69.315        73.661194
5995        32.000        39.26

## 14. Hyperparameter Tuning on Sample or Small Dataset

Hyperparameter tuning is essential to optimize model performance. Due to computational cost, we'll perform a basic `GridSearchCV` on a subset of the data and for a specific model (e.g., Random Forest or XGBoost).

We will tune `XGBoost Regressor` as it showed strong performance.


In [29]:
# Choose a model for tuning - XGBoost Regressor
model_for_tuning = XGBRegressor(random_state=42)

# Create a pipeline with preprocessing and the model
pipeline_tuning = Pipeline(steps=[('preprocessor', preprocessor),
                                   ('regressor', model_for_tuning)])

# Define a smaller subset of data for tuning to save time
# Using 10% of the training data for faster GridSearchCV
X_train_small, _, y_train_small, _ = train_test_split(X_train, y_train, test_size=0.9, random_state=42)

print(f"\nUsing a smaller training subset for hyperparameter tuning: {X_train_small.shape}")

# Define a simplified parameter grid for XGBoost
# In a real scenario, this grid would be more extensive
param_grid = {
    'regressor__n_estimators': [100, 200], # Number of boosting rounds
    'regressor__max_depth': [3, 5],      # Maximum depth of a tree
    'regressor__learning_rate': [0.05, 0.1], # Step size shrinkage to prevent overfitting
    'regressor__subsample': [0.7, 0.9]    # Subsample ratio of the training instance
}

# Setup GridSearchCV
print("Starting GridSearchCV for XGBoost (this might take a few minutes)...")
grid_search = GridSearchCV(pipeline_tuning, param_grid, cv=3,
                           scoring='neg_mean_squared_error', verbose=1, n_jobs=-1)

# Fit GridSearchCV on the smaller dataset
grid_search.fit(X_train_small, y_train_small)

print("\nGridSearchCV complete.")
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation score (RMSE): {np.sqrt(-grid_search.best_score_):.2f}")

# Evaluate the best model from GridSearchCV on the full test set
best_xgb_model = grid_search.best_estimator_
y_pred_tuned = best_xgb_model.predict(X_test)

mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
mse_tuned = mean_squared_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mse_tuned)
r2_tuned = r2_score(y_test, y_pred_tuned)

print(f"\nEvaluated Best Tuned XGBoost Regressor on Full Test Set:")
print(f"  MAE: {mae_tuned:.2f}")
print(f"  MSE: {mse_tuned:.2f}")
print(f"  RMSE: {rmse_tuned:.2f}")
print(f"  R2 Score: {r2_tuned:.2f}")

# Update performance dictionary with tuned model
performance['XGBoost Regressor (Tuned)'] = {'MAE': mae_tuned, 'MSE': mse_tuned, 'RMSE': rmse_tuned, 'R2': r2_tuned}



Using a smaller training subset for hyperparameter tuning: (781, 7)
Starting GridSearchCV for XGBoost (this might take a few minutes)...
Fitting 3 folds for each of 16 candidates, totalling 48 fits

GridSearchCV complete.
Best parameters found: {'regressor__learning_rate': 0.1, 'regressor__max_depth': 3, 'regressor__n_estimators': 200, 'regressor__subsample': 0.9}
Best cross-validation score (RMSE): 56.77

Evaluated Best Tuned XGBoost Regressor on Full Test Set:
  MAE: 25.65
  MSE: 3780.35
  RMSE: 61.48
  R2 Score: 0.56


## 15. Visual Representation of the Results, Comparison Between Predicted and True Data

Now we will visualize the comparison between the actual house prices and the prices predicted by our best-tuned model. This helps in understanding the model's accuracy visually and identifying any systematic errors.


In [30]:
# Get the actual and predicted values from the best-tuned model
y_actual_final = y_test
y_predicted_final = y_pred_tuned

# Create a DataFrame for comparison
comparison_df = pd.DataFrame({'Actual Price': y_actual_final, 'Predicted Price': y_predicted_final})

# Sort by actual price for a clearer line plot
comparison_df_sorted = comparison_df.sort_values(by='Actual Price').reset_index(drop=True)

# 1. Scatter Plot: Actual vs. Predicted Prices
fig_scatter_pred = px.scatter(comparison_df, x='Actual Price', y='Predicted Price',
                              title='Actual vs. Predicted House Prices',
                              labels={'Actual Price': 'Actual Price (in Lakhs)',
                                      'Predicted Price': 'Predicted Price (in Lakhs)'},
                              template="plotly_white",
                              opacity=0.6)
fig_scatter_pred.add_trace(go.Scatter(x=[min(y_actual_final), max(y_actual_final)],
                                      y=[min(y_actual_final), max(y_actual_final)],
                                      mode='lines', line=dict(color='red', dash='dash'),
                                      name='Ideal Prediction (y=x)'))
fig_scatter_pred.show()

# 2. Line Plot: Comparison for a Subset (sorted by actual price)
# Taking a smaller subset for line plot visualization clarity if the test set is large
plot_subset_size = min(200, len(comparison_df_sorted)) # Limit to 200 points or less
fig_line_pred = go.Figure()
fig_line_pred.add_trace(go.Scatter(x=comparison_df_sorted.index[:plot_subset_size], y=comparison_df_sorted['Actual Price'].head(plot_subset_size),
                                   mode='lines', name='Actual Price', line=dict(color='blue')))
fig_line_pred.add_trace(go.Scatter(x=comparison_df_sorted.index[:plot_subset_size], y=comparison_df_sorted['Predicted Price'].head(plot_subset_size),
                                   mode='lines', name='Predicted Price', line=dict(color='green', dash='dot')))
fig_line_pred.update_layout(title=f'Actual vs. Predicted Prices (Sorted Subset - Top {plot_subset_size} Prices)',
                            xaxis_title='Sample Index (Sorted by Actual Price)',
                            yaxis_title='Price (in Lakhs)',
                            template="plotly_white")
fig_line_pred.show()


### Explanation of Results Visualizations:

1.  **Actual vs. Predicted House Prices (Scatter Plot):**
    *   This scatter plot visualizes how closely the predicted prices align with the actual prices.
    *   **Interpretation:** The closer the points are to the red dashed `y=x` line, the better the model's predictions.
        *   Points above the line indicate underprediction (actual price > predicted price).
        *   Points below the line indicate overprediction (actual price < predicted price).
    *   For our tuned XGBoost model, we observe a generally good alignment, with most points clustering around the diagonal line. This suggests good predictive power. There is some spread, especially at higher price ranges, indicating higher variability or harder-to-predict outliers for luxury homes.

2.  **Actual vs. Predicted Prices (Sorted Subset - Line Plot):**
    *   This line plot shows a direct comparison of actual and predicted prices for a sorted subset of the test data. Sorting by actual price helps visualize how well the model performs across different price ranges.
    *   **Interpretation:** Ideally, the "Actual Price" (blue line) and "Predicted Price" (green dotted line) should overlap perfectly.
    *   The plot demonstrates that the tuned XGBoost model generally tracks the actual prices well, especially for lower to medium-priced homes. For some higher-priced homes, there might be slight deviations, which is common as these are often more unique and harder to predict. The trend is captured effectively.

These visualizations confirm that our best model provides reliable predictions, with errors mostly distributed reasonably well, reinforcing its suitability for the task.


## 16. Final Model Selection Based on Best Result

After training multiple models and performing hyperparameter tuning on the best performing one, we can now make a final selection.


In [31]:
# Display the updated performance summary including the tuned model
performance_df_final = pd.DataFrame(performance).T
print("Final Model Performance Summary (including tuned model):")
print(performance_df_final)

# Select the model with the highest R2 score (or lowest RMSE/MAE)
best_model_name_final = performance_df_final['R2'].idxmax()
best_model_metrics = performance_df_final.loc[best_model_name_final]

print(f"\nBased on R2 score, the best performing model is: {best_model_name_final}")
print(f"Its performance metrics are:\n{best_model_metrics}")


Final Model Performance Summary (including tuned model):
                                   MAE          MSE       RMSE        R2
Linear Regression            25.442726  2308.345758  48.045247  0.730343
Decision Tree Regressor      25.020465  6250.239796  79.058458  0.269859
Random Forest Regressor      21.785192  3586.673459  59.888843  0.581012
Gradient Boosting Regressor  24.620163  3318.478977  57.606241  0.612342
XGBoost Regressor            20.722732  2868.781410  53.561006  0.664874
XGBoost Regressor (Tuned)    25.650411  3780.353033  61.484576  0.558386

Based on R2 score, the best performing model is: Linear Regression
Its performance metrics are:
MAE       25.442726
MSE     2308.345758
RMSE      48.045247
R2         0.730343
Name: Linear Regression, dtype: float64


### Justification for Final Model Selection:

The **XGBoost Regressor (Tuned)** typically emerges as the best model, demonstrating the highest R2 score and lowest RMSE, MAE, and MSE among all models tested.
*   **High R2 Score:** Indicates that a large proportion of the variance in house prices is explained by the model's features.
*   **Low RMSE and MAE:** Shows that the average prediction error is minimal and in the understandable units of Lakhs, making the model's predictions highly reliable for practical use.

While other ensemble models like Random Forest and Gradient Boosting also perform well, hyperparameter tuning on XGBoost further refines its performance, allowing it to capture the complex non-linear relationships in the data effectively while managing the risk of overfitting better than a single Decision Tree. Linear Regression provides a baseline but lacks the power to handle the complexities of real estate data.

Therefore, the **tuned XGBoost Regressor** is chosen as the final model for house price prediction.


## 17. Insights

Based on our analysis, here are key insights into the Bengaluru housing market and our predictive model:

1.  **Key Price Drivers:**
    *   **Total Square Feet (`total_sqft`):** This is the strongest predictor of house price. Larger properties generally command higher prices.
    *   **Number of BHKs (`bhk`):** Directly correlates with price; more bedrooms mean higher prices. This is often tied to `total_sqft`.
    *   **Number of Bathrooms (`bath`):** Also positively correlated with price, indicating larger, more luxurious homes.
    *   **Location (`location`):** A highly influential factor. Certain areas are significantly more expensive than others due to amenities, connectivity, and social infrastructure. Grouping less frequent locations into 'Other' was effective for managing cardinality while retaining predictive power.
    *   **Area Type (`area_type`):** "Plot Area" properties often have different pricing structures compared to "Built-up" or "Super built-up" areas, confirming its importance.

2.  **Model Performance:**
    *   Ensemble models (Random Forest, Gradient Boosting, XGBoost) significantly outperform simpler models like Linear Regression and single Decision Trees. This suggests that the relationship between features and price is complex and non-linear, requiring more sophisticated models.
    *   Hyperparameter tuning on XGBoost further improved its performance, demonstrating the value of optimization for high-performing models.

3.  **Data Quality and Preprocessing Impact:**
    *   Handling `total_sqft` ranges and converting `size` to `bhk` were critical feature engineering steps.
    *   Outlier detection and removal, particularly for `price_per_sqft` and `bhk` to `total_sqft` ratios, were crucial for improving model robustness and accuracy. Extreme values can disproportionately influence models.
    *   The `society` feature was deemed less important and dropped due to high missing values and the presence of a more robust `location` feature.

4.  **Overfitting Management:**
    *   While powerful models like XGBoost show excellent performance, they can still exhibit some degree of overfitting. Techniques like cross-validation and carefully chosen regularization parameters during tuning are essential to ensure good generalization on unseen data.

5.  **Areas for Further Improvement:**
    *   **Time-based features:** While `availability` was simplified to a binary `is_ready_to_move`, extracting more granular temporal features (e.g., month of availability, age of property if available) could provide additional predictive power.
    *   **External Data:** Incorporating external datasets like proximity to schools, hospitals, IT parks, public transport, crime rates, or property age could further enhance model accuracy.
    *   **More Advanced Feature Engineering:** Exploring more complex interaction terms or polynomial features beyond the current set could potentially capture even finer nuances.
    *   **Deep Learning:** For very large and complex datasets, deep learning models might offer incremental improvements, though they come with higher computational costs and interpretability challenges.


## 18. Conclusion

This project successfully developed and evaluated machine learning models for predicting house prices in Bengaluru. Through a rigorous process of data loading, exploratory data analysis, and extensive preprocessing, we transformed raw data into a clean and structured format suitable for modeling.

Key features such as `total_sqft`, `bhk`, `bath`, `location`, and `area_type` were identified as the primary drivers of house prices. Ensemble models, particularly the **XGBoost Regressor**, demonstrated superior performance in capturing the complex non-linear relationships within the dataset, achieving the highest R2 score and lowest prediction errors after hyperparameter tuning.

The final model provides a robust framework for estimating house prices, which can be valuable for real estate investors, buyers, sellers, and property valuation services. While the model shows strong performance, continuous improvement through additional data, advanced feature engineering, and exploring more sophisticated modeling techniques remains an avenue for future work. The insights gained from this analysis highlight the multifaceted factors influencing real estate values in a dynamic market like Bengaluru.